# 🐸 Großer Lauf: Pretrain + Fine-Tune (für „Save & Run All“)

Dieses Notebook enthält nur den großen Lauf, damit man es mit **Save Version → Save & Run All (Commit)** im Hintergrund laufen lassen kann (Tab darf zu sein). Ergebnisse danach unter *Output* der Version: `big/pre/…`, `big/ft/…` mit Modellen, Sample-Rastern, `loss.png`, `metrics.csv`.

Vorher: GPU (T4) und Internet an, Dataset(s) mit den Ordnern `wednesday-frogs/` und `frog-photos/` als Input hinzufügen.
Die Parameter stehen in der nächsten Code-Zelle nach dem Setup – nach dem Tempo-Test anpassen.

In [ ]:
import os
REPO = "https://github.com/alexanderulbrich/wednesday.git"
# Immer mit festen Pfaden arbeiten – so darf diese Zelle beliebig oft laufen (erstes Mal: klonen, danach: aktualisieren).
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
%cd {WORK}
!git clone -q $REPO wednesday 2>/dev/null || git -C wednesday pull -q
%cd {WORK}/wednesday
!pip install -q -e .
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os

def find_dataset(name):
    """Der Ordner name/128 mit den Bildern – egal wo Kaggle/Colab ihn einhängt.

    Der Ordner direkt über 128/ muss exakt `name` heißen. So klappt es mit zwei getrennten Datasets
    (/kaggle/input/frog-photos/128) genauso wie mit beiden Zips in einem gemeinsamen Dataset
    (/kaggle/input/irgendwas/frog-photos/128).
    """
    hits = sorted(glob.glob(f"{ROOT}/**/{name}/128", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{name}/128' nicht gefunden unter {ROOT} – Dataset als Input hinzugefügt? "
                                f"Inhalt prüfen mit: !find {ROOT} -maxdepth 5 -type d")
    if len(hits) > 1:
        print(f"Achtung, mehrere Treffer für {name}, nehme den ersten: {hits}")
    return hits[0]

if os.path.exists("/kaggle"):
    ROOT, OUT = "/kaggle/input", "/kaggle/working"
else:  # Colab
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, OUT = "/content/data", "/content/drive/MyDrive/wednesday-runs"   # Runs auf Drive -> überleben Timeouts
    for z in ("wednesday-frogs", "frog-photos"):
        !mkdir -p {ROOT}/{z} && unzip -qo /content/drive/MyDrive/{z}.zip -d {ROOT}/{z}

MEMES, PHOTOS = find_dataset("wednesday-frogs"), find_dataset("frog-photos")
RES = 64   # trainiert wird in 64 px, Quelle sind die schärferen 128er Bilder
for d in (MEMES, PHOTOS):
    print(d, len(os.listdir(d)), "Bilder")

In [ ]:
# ---- Parameter (Zeiten in Minuten; die Schrittzahl ergibt sich aus dem gemessenen Tempo) ----
BASE = 64        # Netzgröße: 64 ≈ 18M Parameter, 48 ≈ 10M, 32 ≈ 4,5M
PRE_MIN = 180    # Pretraining auf Fotos + Memes
FT_MIN = 30      # Fine-Tune nur auf Memes, behält den Stand mit dem besten Meme-Validation-Loss
RUN = f"{OUT}/big"
print(BASE, PRE_MIN, FT_MIN, RUN)

## Stufe 1: Pretrain

In [ ]:
!python train.py --data {PHOTOS},{MEMES} --res {RES} --base {BASE} --minutes {PRE_MIN} --steps 1000000 --lr 2e-4 --warmup 1000 --batch 64 --out {RUN}/pre --sample-every 2000 --val-every 1000 --save-every 2000 --keep-best val

## Stufe 2: Fine-Tune (bester Stand nach Meme-Validierung → `best_frog.pt`)

In [ ]:
!python train.py --init {RUN}/pre/last.pt --data {MEMES} --res {RES} --minutes {FT_MIN} --steps 1000000 --lr 1e-4 --warmup 200 --batch 64 --out {RUN}/ft --sample-every 500 --val-every 250 --save-every 1000 --keep-best wednesday-frogs

## Ergebnis

In [ ]:
from IPython.display import Image, display
for stage, ckpt in (("pre", "frog.pt"), ("ft", "best_frog.pt")):
    !python generate.py --ckpt {RUN}/{stage}/{ckpt} --n 16 --grid --size 128 --seed 42 --no-open --out {RUN}/{stage}/generated
    print(stage); display(Image(f"{RUN}/{stage}/loss.png"))
    display(Image(sorted(glob.glob(f"{RUN}/{stage}/generated/*.png"))[-1]))